# EDA — BANKING77 (train)

Cel: poznać dane, z którymi pracujemy — rozkład klas, długości, duplikaty, niejednoznaczne intencje — i zapisać wnioski, które pójdą do README (raport) oraz do decyzji o modelach (`class_weight`, `max_len`).

**Zasady:**
- tylko `train.csv` przez `load_train()`; test jest zamknięty do oceny końcowej w W3 — w tym notebooku **nigdy** `load_test()` (strażnik nie skanuje notebooków, tę granicę pilnujemy sami);
- nie edytujemy plików CSV (suma SHA-256);
- przed commitem: *Kernel → Restart Kernel and Clear Outputs of All Cells* — w repo bez surowych tekstów poza pojedynczymi przykładami.

Szkielet: asystent (Claude), 5.10.2026 · analizy i wnioski: Marcin.

## 0. Wczytanie danych

In [ ]:
import json

import pandas as pd

from banking77_intent_ml.config import load_settings
from banking77_intent_ml.data.labels import load_label_map
from banking77_intent_ml.data.loader import load_train
from banking77_intent_ml.data.splits import VAL_IDS_FILE, normalize_text  # noqa: F401 (sekcja 5)

pd.set_option("display.max_colwidth", 200)  # pełne teksty wiadomości w tabelach

settings = load_settings()
train = load_train(settings)  # kolumny: text, category; indeks = pozycja wiersza w train.csv
label_map = load_label_map(settings.dataset.label_map_path)  # 77 kategorii -> id
val_ids = json.loads((settings.paths.splits_dir / VAL_IDS_FILE).read_text(encoding="utf-8"))
is_val = train.index.isin(val_ids)  # True = wiersz w części walidacyjnej zamrożonego splitu

print(train.shape, "| kategorii:", train["category"].nunique(), "| w walidacji:", int(is_val.sum()))
train.head()

## 1. Ogólny obraz i braki

Kształt, kolumny, typy. Braki: loader gwarantuje brak pustych tekstów i kategorii (inaczej dane by się nie wczytały). Rubryka wymaga „braki i ich powody” — tu: zero braków + granica wiedzy (autorzy zbioru nie dokumentują czyszczenia; README → *Dane*).

In [ ]:
# Twój kod: np. train.info(), train.describe(include="all")


**Wniosek (Marcin):** …

## 2. Rozkład klas

Train jest niezrównoważony (35–187 na klasę), test ma dokładnie 40 na klasę (wiemy z audytu integralności, nie z oglądania testu). Pytania: które klasy są najmniej liczne? Czy niezrównoważenie może zaszkodzić macro-F1 na walidacji? (decyzja o `class_weight` — przed N1).

In [ ]:
# Twój kod: liczności kategorii i wykres słupkowy (wszystkie 77 klas)


**Wniosek (Marcin):** …

## 3. Przykłady i rodziny podobnych intencji

Losowe teksty z kilku kategorii, potem rodziny po początku nazwy kategorii: `card_*`, `top_up_*`, `pending_*`, `transfer_*`, `*_not_working` — tam spodziewamy się pomyłek modeli. Z podglądu: „Give me my card, I need it now!” ma etykietę `card_delivery_estimate`, choć pasowałby też do `card_arrival`.

In [ ]:
# Twój kod: próbki tekstów z wybranych kategorii (seed z configu: settings.split.random_state)


**Wniosek (Marcin):** …

## 4. Długości tekstów (znaki, słowa, tokeny)

Znaki i słowa — teraz. Tokeny DistilBERT (95./99./100. percentyl → `max_len`, w configu 128) — sob 10.10, po przypięciu rewizji wag zamiast `PIN-ME`.

In [ ]:
# Twój kod: długości w znakach i słowach, histogram, najkrótsze i najdłuższe teksty


In [ ]:
# Sob 10.10: długości w tokenach (tokenizer DistilBERT z przypiętą rewizją z configu)


**Wniosek (Marcin):** …

## 5. Duplikaty po normalizacji i konflikt etykiet

Normalizacja z configu — `normalize_text` (NFKC → casefold → tokeny `\w+`), ta sama, której używa split. Z manifestu: 9 972 grupy (31 nadmiarowych duplikatów); grupa konfliktowa: wiersze 2002 i 3160 (`pending_top_up` / `top_up_reverted`) — w train, etykiet nie poprawiamy.

In [ ]:
# Twój kod: klucz = normalize_text(tekst); grupy z > 1 wierszem; wiersze 2002 i 3160


**Wniosek (Marcin):** …

## 6. Teksty skrajne i niejednoznaczne (10–20 z train)

Najkrótsze, najdłuższe i takie, przy których sam masz wątpliwość co do etykiety. Ying & Thomas (2022) wskazują automatycznie ~14% potencjalnych problemów etykiet w train — nie poprawiamy ich, opisujemy.

In [ ]:
# Twój kod: wybór 10–20 przykładów (tylko z train)


**Wniosek (Marcin):** …

## 7. Słownictwo (zapowiedź TF-IDF)

Najczęstsze słowa w kilku klasach, zwłaszcza w rodzinach podobnych intencji z sekcji 3 — czy słowa je rozróżniają?

In [ ]:
# Twój kod: najczęstsze słowa w wybranych klasach


**Wniosek (Marcin):** …

## 8. Dane osobowe (PII) — kontrola na train

Audyt z 30.09 (5 wzorców: e-mail, URL, ≥ 9 cyfr, IBAN, przedstawianie się) dał 0 trafień — to nie dowodzi braku PII. Wykrycie istotnych PII = stop i decyzja o zbiorze. Wypisuj tylko liczby trafień i pozycje, nie teksty.

In [ ]:
# Twój kod: wyszukanie wzorców PII (regex) — liczby trafień i pozycje wierszy


**Wniosek (Marcin):** …

## 9. Podsumowanie EDA (do README)

3–5 zdań: co wiemy o danych i co z tego wynika dla protokołu i modeli.

**Wnioski (Marcin):** …